# 03 · Evaluation check (Step 2.2)

**Runtime: T4 GPU** (Runtime → Change runtime type → **T4 GPU**).

Checks the evaluation pipeline end to end on **100 test examples per task** with the untrained base model:
1. Pinned environment (same as before).
2. Checks the label files on Drive match the committed hashes, and unzips the test photos.
3. Runs `run_eval.py`: shared image preparation → chat prompt → vLLM greedy decoding → metrics.

Expect about **20–30 minutes** (most of it is the one-time 9 GB model download and vLLM start-up). These 100-example numbers are only a pipeline check; the real baseline (E1) is Step 2.3 on the full test set.

Every command runs through `sh()`, which stops with an error if the command fails.

## 1 · Settings, Drive, code

In [ ]:
import os
import subprocess

from google.colab import drive

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
BRANCH = "main"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
DATA = f"{DRIVE_ROOT}/datasets"
LOCAL = "/content/data"
RUNS = f"{DRIVE_ROOT}/runs"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"
RUN_ID = "e1-check-t4"


def sh(cmd, **env):
    """Run a shell command, stream its output, and raise if it fails."""
    proc = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, **env},
    )
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise RuntimeError(
            f"Command failed (exit {proc.returncode}). Stop here and paste the output above."
        )


drive.mount("/content/drive")
sh(
    f"if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}; else git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}; fi"
)
sh(f"git -C {REPO_DIR} log --oneline -1")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

**Paste back:** the commit line and the GPU line (should say Tesla T4).

## 2 · Pinned environment (~3–5 min)

In [ ]:
%%time
sh("pip install -q uv")
sh(f"uv venv -q --clear --python 3.13 {VENV}")
sh(
    f"uv pip install -q --python {PY} -e {REPO_DIR} -r {REPO_DIR}/requirements/colab-gpu.txt -c {REPO_DIR}/requirements/colab-gpu.lock.txt"
)
sh(f"{PY} -m vizwiz.compat versions")

## 3 · Check label files, unzip test photos (~1 min)
The test split is the official val set, so only `val.zip` is needed.

In [ ]:
import hashlib
import json

expected = json.load(open(f"{REPO_DIR}/results/splits/files.json"))
for name in ("vqa_test.jsonl", "quality_test.jsonl"):
    digest = hashlib.sha256(open(f"{DATA}/processed/{name}", "rb").read()).hexdigest()
    assert digest == expected[name]["sha256"], f"{name} differs from the committed hash"
    print(f"{name}: matches committed hash")
sh(f"{PY} -m vizwiz.data.download extract --root {DATA} --dest {LOCAL} --splits val")

## 4 · Run the evaluation on 100 examples per task (~15–20 min)
Downloads the model (~9 GB) to local disk, starts vLLM, then runs the three tasks.

In [ ]:
%%time
sh(
    f"{PY} -m vizwiz.eval.run_eval --records {DATA}/processed --images {LOCAL}/images --split test --limit 100 --run-id {RUN_ID} --out {RUNS}/{RUN_ID}",
    HF_HOME="/content/hf_cache",
)

**Paste back:** the `answerability score uses first tokens ...` line, the three lines that start with `[answer] vqa_accuracy`, `[answerability] average_precision`, `[quality] macro_f1`, and any warning or error.

## 5 · Look at a few model outputs (stays in this notebook, do not paste)
A quick sanity check that the outputs look like answers. Prints only the model's replies, not questions or photos.

In [ ]:
for task in ("answer", "answerability", "quality"):
    rows = [json.loads(line) for line in open(f"{RUNS}/{RUN_ID}/predictions_{task}.jsonl")][:5]
    print(task, [r["prediction"] for r in rows])

**Paste back:** nothing from this cell. Just tell me if the replies look like short answers, one word, and JSON lists respectively.

## 6 · Flush Drive, then disconnect
Upload back from `MyDrive/vizwiz/runs/e1-check-t4/`: **`metrics.json`** only (summary numbers; the prediction files stay on Drive).

Then Runtime → Disconnect and delete runtime.

In [ ]:
drive.flush_and_unmount()
print("Drive flushed. Safe to disconnect.")